# Spaceship Titanic v2 — 8-Model Stacking Ensemble

**Goal:** Beat 0.80827 LB (best from v1 notebook) by adding diverse non-GBM models and stacking them.

**Models:**
1. XGBoost (Optuna-tuned)
2. LightGBM (Optuna-tuned)
3. CatBoost
4. TabNet (deep learning for tabular data)
5. ExtraTrees (Optuna-tuned)
6. Random Forest (Optuna-tuned)
7. SVM RBF (Optuna-tuned)
8. KNN (Optuna-tuned)

**Ensemble:** Level-2 Logistic Regression stacking on out-of-fold predictions + threshold optimization.

---
## 1. Setup

In [ ]:
!pip install xgboost lightgbm catboost pytorch_tabnet optuna -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from pytorch_tabnet.tab_model import TabNetClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
np.random.seed(SEED)

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

print('\u2713 All imports loaded')

---
## 2. Data Loading

In [ ]:
import kagglehub
path = kagglehub.competition_download('spaceship-titanic')
DATA_PATH = Path(path)

train_raw = pd.read_csv(DATA_PATH / 'train.csv')
test_raw = pd.read_csv(DATA_PATH / 'test.csv')

print(f'Train: {train_raw.shape}, Test: {test_raw.shape}')

---
## 3. Feature Engineering & Preprocessing

Reuses the proven pipeline from v1 (40+ features with interaction features).

In [ ]:
train_ids = train_raw['PassengerId'].copy()
test_ids = test_raw['PassengerId'].copy()
target = train_raw['Transported'].copy()

train_raw['is_train'] = True
test_raw['is_train'] = False
test_raw['Transported'] = np.nan

df = pd.concat([train_raw, test_raw], axis=0, ignore_index=True)

spending_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

# --- Group ---
df['Group'] = df['PassengerId'].apply(lambda x: x.split('_')[0])
grp = df.groupby('Group').size().reset_index(name='GroupSize')
df = df.merge(grp, on='Group', how='left')
df['IsAlone'] = (df['GroupSize'] == 1).astype(int)

# --- Cabin ---
cabin_parts = df['Cabin'].str.split('/', expand=True)
cabin_parts.columns = ['Deck', 'CabinNum', 'Side']
cabin_parts['CabinNum'] = pd.to_numeric(cabin_parts['CabinNum'], errors='coerce')
df = pd.concat([df, cabin_parts], axis=1)

# --- Spending ---
df['TotalSpending'] = df[spending_cols].sum(axis=1, skipna=True)
df['SpendingCategories'] = (df[spending_cols] > 0).sum(axis=1)
df['HasSpending'] = (df['TotalSpending'] > 0).astype(int)
for col in spending_cols:
    df[f'{col}_log'] = np.log1p(df[col])
df['TotalSpending_log'] = np.log1p(df['TotalSpending'])

# --- Age ---
df['IsChild'] = (df['Age'] < 18).astype(float)
df['IsElderly'] = (df['Age'] > 60).astype(float)
df['AgeGroup'] = pd.cut(df['Age'], bins=[0, 12, 18, 30, 50, 100],
                         labels=['Child', 'Teen', 'YoungAdult', 'Adult', 'Senior'])

# --- Family ---
df['FamilyName'] = df['Name'].apply(lambda x: x.split()[-1] if pd.notna(x) else np.nan)
fam = df.groupby('FamilyName').size().reset_index(name='FamilySize')
df = df.merge(fam, on='FamilyName', how='left')
df['FamilySize'] = df['FamilySize'].fillna(1)

# --- Imputation ---
has_spending = df['TotalSpending'] > 0
df.loc[has_spending & df['CryoSleep'].isna(), 'CryoSleep'] = False
df['CryoSleep'] = df['CryoSleep'].fillna(df['CryoSleep'].mode()[0])

for col in spending_cols:
    df.loc[df['CryoSleep'] == True, col] = df.loc[df['CryoSleep'] == True, col].fillna(0)
    median_val = df.loc[df['CryoSleep'] == False, col].median()
    df[col] = df[col].fillna(median_val)

df['Age'] = df['Age'].fillna(df['Age'].median())
df['CabinNum'] = df['CabinNum'].fillna(df['CabinNum'].median())
for col in ['HomePlanet', 'Destination', 'VIP', 'Deck', 'Side']:
    df[col] = df[col].fillna(df[col].mode()[0])

# Recalculate after imputation
df['TotalSpending'] = df[spending_cols].sum(axis=1)
df['SpendingCategories'] = (df[spending_cols] > 0).sum(axis=1)
df['HasSpending'] = (df['TotalSpending'] > 0).astype(int)
df['TotalSpending_log'] = np.log1p(df['TotalSpending'])
for col in spending_cols:
    df[f'{col}_log'] = np.log1p(df[col])
df['IsChild'] = (df['Age'] < 18).astype(int)
df['IsElderly'] = (df['Age'] > 60).astype(int)
df['AgeGroup'] = pd.cut(df['Age'], bins=[0, 12, 18, 30, 50, 100],
                         labels=['Child', 'Teen', 'YoungAdult', 'Adult', 'Senior'])

# === INTERACTION FEATURES ===
luxury = df['Spa'] + df['VRDeck'] + df['RoomService']
basic = df['FoodCourt'] + df['ShoppingMall']
df['LuxurySpending_log'] = np.log1p(luxury)
df['BasicSpending_log'] = np.log1p(basic)
df['LuxuryRatio'] = luxury / (df['TotalSpending'] + 1)

df['CryoSleep'] = df['CryoSleep'].astype(int)
df['Cryo_NoSpend'] = ((df['CryoSleep'] == 1) & (df['TotalSpending'] == 0)).astype(int)

df['Age_x_Spending'] = df['Age'] * df['TotalSpending_log']

df['SpendPerGroupMember'] = df['TotalSpending'] / df['GroupSize']
df['SpendPerGroupMember_log'] = np.log1p(df['SpendPerGroupMember'])

df['CabinRegion'] = pd.cut(df['CabinNum'], bins=5, labels=False).fillna(-1).astype(int)

df['VIP'] = df['VIP'].astype(int)
df['VIP_Young'] = ((df['VIP'] == 1) & (df['Age'] < 30)).astype(int)
df['Alone_Cryo'] = (df['IsAlone'] & df['CryoSleep']).astype(int)

print(f'\u2713 Feature engineering complete. Shape: {df.shape}')

# --- Encode ---
cat_cols_encode = ['HomePlanet', 'Destination', 'Deck', 'Side', 'AgeGroup']
df = pd.get_dummies(df, columns=cat_cols_encode, drop_first=True)

# --- Select features ---
drop_cols = ['PassengerId', 'Cabin', 'Name', 'Group', 'FamilyName',
             'Transported', 'is_train', 'GroupNum',
             'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
             'CabinNum']
feat_cols = [c for c in df.columns if c not in drop_cols]

train_df = df[df['is_train'] == True]
test_df = df[df['is_train'] == False]

X = train_df[feat_cols].values
y = target.values.astype(int)
X_test = test_df[feat_cols].values

scaler = StandardScaler()
X_sc = scaler.fit_transform(X)
X_test_sc = scaler.transform(X_test)

print(f'\u2713 Train: {X.shape}, Test: {X_test.shape}, Features: {len(feat_cols)}')

---
## 4. Helper: OOF Training Loop

Reusable function to collect out-of-fold predictions from any sklearn-compatible model.

In [ ]:
def train_oof(name, model_fn, X_tr, y_tr, X_te, use_proba=True):
    """
    Train model with 5-fold CV, return OOF probabilities + test predictions.
    model_fn(fold): returns a fresh model instance for that fold.
    """
    oof = np.zeros(len(X_tr))
    test_preds = np.zeros(len(X_te))
    scores = []

    for fold, (tr_idx, vl_idx) in enumerate(skf.split(X_tr, y_tr)):
        X_f_tr, X_f_vl = X_tr[tr_idx], X_tr[vl_idx]
        y_f_tr, y_f_vl = y_tr[tr_idx], y_tr[vl_idx]

        model = model_fn(fold)
        model.fit(X_f_tr, y_f_tr)

        if use_proba:
            oof[vl_idx] = model.predict_proba(X_f_vl)[:, 1]
            test_preds += model.predict_proba(X_te)[:, 1] / N_FOLDS
        else:
            oof[vl_idx] = model.decision_function(X_f_vl)
            test_preds += model.decision_function(X_te) / N_FOLDS

        acc = accuracy_score(y_f_vl, (oof[vl_idx] > 0.5).astype(int))
        scores.append(acc)
        print(f'  Fold {fold+1}: {acc:.4f}')

    mean_acc = np.mean(scores)
    print(f'\u2713 {name} CV: {mean_acc:.4f} (+/- {np.std(scores):.4f})\n')
    return oof, test_preds, mean_acc

---
## 5. Gradient Boosting Models (XGBoost + LightGBM + CatBoost)

### 5.1 XGBoost — Optuna Tuning

In [ ]:
def xgb_objective(trial):
    params = {
        'n_estimators': 1000,
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'gamma': trial.suggest_float('gamma', 0.0, 1.0),
        'random_state': SEED,
        'eval_metric': 'logloss',
        'early_stopping_rounds': 50
    }
    scores = []
    for tr_idx, vl_idx in skf.split(X_sc, y):
        m = xgb.XGBClassifier(**params)
        m.fit(X_sc[tr_idx], y[tr_idx],
              eval_set=[(X_sc[vl_idx], y[vl_idx])], verbose=False)
        preds = (m.predict_proba(X_sc[vl_idx])[:, 1] > 0.5).astype(int)
        scores.append(accuracy_score(y[vl_idx], preds))
    return np.mean(scores)

xgb_study = optuna.create_study(direction='maximize', study_name='xgb')
xgb_study.optimize(xgb_objective, n_trials=100, show_progress_bar=True)

print(f'\n\u2713 Best XGBoost CV: {xgb_study.best_value:.4f}')
for k, v in xgb_study.best_params.items():
    print(f'  {k}: {v}')

### 5.2 LightGBM — Optuna Tuning

In [ ]:
def lgb_objective(trial):
    params = {
        'n_estimators': 1000,
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 60),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 1.0),
        'random_state': SEED,
        'verbose': -1,
    }
    scores = []
    for tr_idx, vl_idx in skf.split(X_sc, y):
        m = lgb.LGBMClassifier(**params)
        m.fit(X_sc[tr_idx], y[tr_idx], eval_set=[(X_sc[vl_idx], y[vl_idx])],
              callbacks=[lgb.early_stopping(50, verbose=False)])
        preds = (m.predict_proba(X_sc[vl_idx])[:, 1] > 0.5).astype(int)
        scores.append(accuracy_score(y[vl_idx], preds))
    return np.mean(scores)

lgb_study = optuna.create_study(direction='maximize', study_name='lgb')
lgb_study.optimize(lgb_objective, n_trials=150, show_progress_bar=True)

print(f'\n\u2713 Best LightGBM CV: {lgb_study.best_value:.4f}')
for k, v in lgb_study.best_params.items():
    print(f'  {k}: {v}')

### 5.3 Train GBMs with Best Params (OOF)

In [ ]:
# XGBoost OOF
xgb_best = xgb_study.best_params.copy()
xgb_best.update({'n_estimators': 1000, 'random_state': SEED,
                  'eval_metric': 'logloss', 'early_stopping_rounds': 50})

oof_xgb = np.zeros(len(X_sc))
test_xgb = np.zeros(len(X_test_sc))
xgb_scores = []

print('XGBoost (tuned):')
for fold, (tr_idx, vl_idx) in enumerate(skf.split(X_sc, y)):
    m = xgb.XGBClassifier(**xgb_best)
    m.fit(X_sc[tr_idx], y[tr_idx],
          eval_set=[(X_sc[vl_idx], y[vl_idx])], verbose=False)
    oof_xgb[vl_idx] = m.predict_proba(X_sc[vl_idx])[:, 1]
    test_xgb += m.predict_proba(X_test_sc)[:, 1] / N_FOLDS
    acc = accuracy_score(y[vl_idx], (oof_xgb[vl_idx] > 0.5).astype(int))
    xgb_scores.append(acc)
    print(f'  Fold {fold+1}: {acc:.4f}')
print(f'\u2713 XGBoost CV: {np.mean(xgb_scores):.4f}\n')

# LightGBM OOF
lgb_best = lgb_study.best_params.copy()
lgb_best.update({'n_estimators': 1000, 'random_state': SEED, 'verbose': -1})

oof_lgb = np.zeros(len(X_sc))
test_lgb = np.zeros(len(X_test_sc))
lgb_scores = []

print('LightGBM (tuned):')
for fold, (tr_idx, vl_idx) in enumerate(skf.split(X_sc, y)):
    m = lgb.LGBMClassifier(**lgb_best)
    m.fit(X_sc[tr_idx], y[tr_idx], eval_set=[(X_sc[vl_idx], y[vl_idx])],
          callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_lgb[vl_idx] = m.predict_proba(X_sc[vl_idx])[:, 1]
    test_lgb += m.predict_proba(X_test_sc)[:, 1] / N_FOLDS
    acc = accuracy_score(y[vl_idx], (oof_lgb[vl_idx] > 0.5).astype(int))
    lgb_scores.append(acc)
    print(f'  Fold {fold+1}: {acc:.4f}')
print(f'\u2713 LightGBM CV: {np.mean(lgb_scores):.4f}\n')

# CatBoost OOF
oof_cb = np.zeros(len(X_sc))
test_cb = np.zeros(len(X_test_sc))
cb_scores = []

print('CatBoost:')
for fold, (tr_idx, vl_idx) in enumerate(skf.split(X_sc, y)):
    m = CatBoostClassifier(iterations=1000, depth=6, learning_rate=0.05,
                            l2_leaf_reg=3, random_seed=SEED, verbose=0,
                            early_stopping_rounds=50, eval_metric='Accuracy')
    m.fit(X_sc[tr_idx], y[tr_idx], eval_set=(X_sc[vl_idx], y[vl_idx]))
    oof_cb[vl_idx] = m.predict_proba(X_sc[vl_idx])[:, 1]
    test_cb += m.predict_proba(X_test_sc)[:, 1] / N_FOLDS
    acc = accuracy_score(y[vl_idx], (oof_cb[vl_idx] > 0.5).astype(int))
    cb_scores.append(acc)
    print(f'  Fold {fold+1}: {acc:.4f}')
print(f'\u2713 CatBoost CV: {np.mean(cb_scores):.4f}')

---
## 6. TabNet

In [ ]:
import torch
torch.manual_seed(SEED)

oof_tabnet = np.zeros(len(X_sc))
test_tabnet = np.zeros(len(X_test_sc))
tabnet_scores = []

print('TabNet:')
for fold, (tr_idx, vl_idx) in enumerate(skf.split(X_sc, y)):
    X_f_tr, X_f_vl = X_sc[tr_idx], X_sc[vl_idx]
    y_f_tr, y_f_vl = y[tr_idx], y[vl_idx]

    tn = TabNetClassifier(
        n_d=32, n_a=32, n_steps=5,
        gamma=1.5, lambda_sparse=1e-4,
        optimizer_fn=torch.optim.Adam,
        optimizer_params={'lr': 2e-2},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        scheduler_params={'step_size': 30, 'gamma': 0.5},
        mask_type='entmax',
        seed=SEED,
        verbose=0
    )
    tn.fit(
        X_train=X_f_tr, y_train=y_f_tr,
        eval_set=[(X_f_vl, y_f_vl)],
        eval_metric=['accuracy'],
        max_epochs=200,
        patience=30,
        batch_size=256,
        drop_last=False
    )

    oof_tabnet[vl_idx] = tn.predict_proba(X_f_vl)[:, 1]
    test_tabnet += tn.predict_proba(X_test_sc)[:, 1] / N_FOLDS

    acc = accuracy_score(y_f_vl, (oof_tabnet[vl_idx] > 0.5).astype(int))
    tabnet_scores.append(acc)
    print(f'  Fold {fold+1}: {acc:.4f}')

print(f'\u2713 TabNet CV: {np.mean(tabnet_scores):.4f} (+/- {np.std(tabnet_scores):.4f})')

---
## 7. ExtraTrees — Optuna Tuning

In [ ]:
def et_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1500),
        'max_depth': trial.suggest_int('max_depth', 5, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_float('max_features', 0.3, 1.0),
        'random_state': SEED,
        'n_jobs': -1
    }
    scores = []
    for tr_idx, vl_idx in skf.split(X_sc, y):
        m = ExtraTreesClassifier(**params)
        m.fit(X_sc[tr_idx], y[tr_idx])
        preds = (m.predict_proba(X_sc[vl_idx])[:, 1] > 0.5).astype(int)
        scores.append(accuracy_score(y[vl_idx], preds))
    return np.mean(scores)

et_study = optuna.create_study(direction='maximize', study_name='et')
et_study.optimize(et_objective, n_trials=50, show_progress_bar=True)

print(f'\n\u2713 Best ExtraTrees CV: {et_study.best_value:.4f}')
for k, v in et_study.best_params.items():
    print(f'  {k}: {v}')

In [ ]:
# ExtraTrees OOF with best params
et_best = et_study.best_params.copy()
et_best.update({'random_state': SEED, 'n_jobs': -1})

print('ExtraTrees (tuned):')
oof_et, test_et, et_cv = train_oof(
    'ExtraTrees',
    lambda fold: ExtraTreesClassifier(**et_best),
    X_sc, y, X_test_sc
)

---
## 8. Random Forest — Optuna Tuning

In [ ]:
def rf_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1500),
        'max_depth': trial.suggest_int('max_depth', 5, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_float('max_features', 0.3, 1.0),
        'random_state': SEED,
        'n_jobs': -1
    }
    scores = []
    for tr_idx, vl_idx in skf.split(X_sc, y):
        m = RandomForestClassifier(**params)
        m.fit(X_sc[tr_idx], y[tr_idx])
        preds = (m.predict_proba(X_sc[vl_idx])[:, 1] > 0.5).astype(int)
        scores.append(accuracy_score(y[vl_idx], preds))
    return np.mean(scores)

rf_study = optuna.create_study(direction='maximize', study_name='rf')
rf_study.optimize(rf_objective, n_trials=50, show_progress_bar=True)

print(f'\n\u2713 Best Random Forest CV: {rf_study.best_value:.4f}')
for k, v in rf_study.best_params.items():
    print(f'  {k}: {v}')

In [ ]:
# Random Forest OOF with best params
rf_best = rf_study.best_params.copy()
rf_best.update({'random_state': SEED, 'n_jobs': -1})

print('Random Forest (tuned):')
oof_rf, test_rf, rf_cv = train_oof(
    'RandomForest',
    lambda fold: RandomForestClassifier(**rf_best),
    X_sc, y, X_test_sc
)

---
## 9. SVM (RBF Kernel) — Optuna Tuning

In [ ]:
def svm_objective(trial):
    C = trial.suggest_float('C', 0.01, 100.0, log=True)
    gamma = trial.suggest_float('gamma', 1e-4, 1.0, log=True)
    scores = []
    for tr_idx, vl_idx in skf.split(X_sc, y):
        m = SVC(C=C, gamma=gamma, kernel='rbf', probability=True, random_state=SEED)
        m.fit(X_sc[tr_idx], y[tr_idx])
        preds = (m.predict_proba(X_sc[vl_idx])[:, 1] > 0.5).astype(int)
        scores.append(accuracy_score(y[vl_idx], preds))
    return np.mean(scores)

svm_study = optuna.create_study(direction='maximize', study_name='svm')
svm_study.optimize(svm_objective, n_trials=50, show_progress_bar=True)

print(f'\n\u2713 Best SVM CV: {svm_study.best_value:.4f}')
print(f'  C: {svm_study.best_params["C"]:.4f}')
print(f'  gamma: {svm_study.best_params["gamma"]:.6f}')

In [ ]:
# SVM OOF with best params
svm_C = svm_study.best_params['C']
svm_gamma = svm_study.best_params['gamma']

print('SVM RBF (tuned):')
oof_svm, test_svm, svm_cv = train_oof(
    'SVM',
    lambda fold: SVC(C=svm_C, gamma=svm_gamma, kernel='rbf',
                     probability=True, random_state=SEED),
    X_sc, y, X_test_sc
)

---
## 10. KNN — Optuna Tuning

In [ ]:
def knn_objective(trial):
    params = {
        'n_neighbors': trial.suggest_int('n_neighbors', 3, 50),
        'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
        'metric': trial.suggest_categorical('metric', ['euclidean', 'manhattan', 'minkowski']),
        'p': trial.suggest_int('p', 1, 3),
        'n_jobs': -1
    }
    scores = []
    for tr_idx, vl_idx in skf.split(X_sc, y):
        m = KNeighborsClassifier(**params)
        m.fit(X_sc[tr_idx], y[tr_idx])
        preds = (m.predict_proba(X_sc[vl_idx])[:, 1] > 0.5).astype(int)
        scores.append(accuracy_score(y[vl_idx], preds))
    return np.mean(scores)

knn_study = optuna.create_study(direction='maximize', study_name='knn')
knn_study.optimize(knn_objective, n_trials=50, show_progress_bar=True)

print(f'\n\u2713 Best KNN CV: {knn_study.best_value:.4f}')
for k, v in knn_study.best_params.items():
    print(f'  {k}: {v}')

In [ ]:
# KNN OOF with best params
knn_best = knn_study.best_params.copy()
knn_best.update({'n_jobs': -1})

print('KNN (tuned):')
oof_knn, test_knn, knn_cv = train_oof(
    'KNN',
    lambda fold: KNeighborsClassifier(**knn_best),
    X_sc, y, X_test_sc
)

---
## 11. Model Comparison

In [ ]:
# Collect all results
model_names = ['XGBoost', 'LightGBM', 'CatBoost', 'TabNet',
               'ExtraTrees', 'RandomForest', 'SVM', 'KNN']
oof_list = [oof_xgb, oof_lgb, oof_cb, oof_tabnet,
            oof_et, oof_rf, oof_svm, oof_knn]
test_list = [test_xgb, test_lgb, test_cb, test_tabnet,
             test_et, test_rf, test_svm, test_knn]

print('=' * 40)
print('MODEL COMPARISON (5-Fold CV Accuracy)')
print('=' * 40)
cv_scores = {}
for name, oof in zip(model_names, oof_list):
    acc = accuracy_score(y, (oof > 0.5).astype(int))
    cv_scores[name] = acc
    print(f'  {name:15s}: {acc:.4f}')

# Simple average
avg_oof = np.mean(oof_list, axis=0)
avg_acc = accuracy_score(y, (avg_oof > 0.5).astype(int))
print(f'  {"Simple Avg":15s}: {avg_acc:.4f}')

# Best individual
best_name = max(cv_scores, key=cv_scores.get)
print(f'\nBest single model: {best_name} ({cv_scores[best_name]:.4f})')

---
## 12. Stacking Meta-Model

Train Logistic Regression on the 8-column OOF prediction matrix. Use 5-fold CV on the meta-features for an honest stacking score.

In [ ]:
# Build stacking matrices
stack_train = np.column_stack(oof_list)   # (n_train, 8)
stack_test = np.column_stack(test_list)    # (n_test, 8)

print(f'Stack train shape: {stack_train.shape}')
print(f'Stack test shape:  {stack_test.shape}')

# --- 5-fold CV on meta-model for honest score ---
meta_oof = np.zeros(len(y))
meta_test_preds = np.zeros(len(X_test_sc))
meta_scores = []

print('\nStacking (LogReg meta-model):')
for fold, (tr_idx, vl_idx) in enumerate(skf.split(stack_train, y)):
    meta = LogisticRegression(C=1.0, random_state=SEED, max_iter=1000)
    meta.fit(stack_train[tr_idx], y[tr_idx])

    meta_oof[vl_idx] = meta.predict_proba(stack_train[vl_idx])[:, 1]
    meta_test_preds += meta.predict_proba(stack_test)[:, 1] / N_FOLDS

    acc = accuracy_score(y[vl_idx], (meta_oof[vl_idx] > 0.5).astype(int))
    meta_scores.append(acc)
    print(f'  Fold {fold+1}: {acc:.4f}')

stack_cv = np.mean(meta_scores)
print(f'\u2713 Stacking CV: {stack_cv:.4f} (+/- {np.std(meta_scores):.4f})')

# Also train final meta-model on all OOF data
final_meta = LogisticRegression(C=1.0, random_state=SEED, max_iter=1000)
final_meta.fit(stack_train, y)
final_meta_test = final_meta.predict_proba(stack_test)[:, 1]

print(f'\nMeta-model weights: {dict(zip(model_names, final_meta.coef_[0].round(3)))}')

---
## 13. Threshold Optimization

Search for the best threshold on OOF predictions instead of hardcoded 0.5.

In [ ]:
# Search thresholds on stacking OOF
thresholds = np.arange(0.40, 0.61, 0.005)
thresh_results = []

for t in thresholds:
    acc = accuracy_score(y, (meta_oof > t).astype(int))
    thresh_results.append((t, acc))

thresh_df = pd.DataFrame(thresh_results, columns=['Threshold', 'Accuracy'])
best_thresh = thresh_df.loc[thresh_df['Accuracy'].idxmax()]

print(f'Best threshold: {best_thresh["Threshold"]:.3f} -> Accuracy: {best_thresh["Accuracy"]:.4f}')
print(f'Default (0.5):  -> Accuracy: {accuracy_score(y, (meta_oof > 0.5).astype(int)):.4f}')

plt.figure(figsize=(10, 4))
plt.plot(thresh_df['Threshold'], thresh_df['Accuracy'], 'b-o', markersize=3)
plt.axvline(x=best_thresh['Threshold'], color='red', linestyle='--',
            label=f'Best: {best_thresh["Threshold"]:.3f}')
plt.axvline(x=0.5, color='gray', linestyle=':', label='Default: 0.5')
plt.xlabel('Threshold')
plt.ylabel('OOF Accuracy')
plt.title('Threshold Optimization (Stacking)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

BEST_THRESH = best_thresh['Threshold']

---
## 14. Generate Submissions

In [ ]:
def save_submission(name, preds_proba, threshold=0.5):
    sub = pd.DataFrame({
        'PassengerId': test_ids,
        'Transported': (preds_proba > threshold)
    })
    fname = f'submission_{name}.csv'
    sub.to_csv(fname, index=False)
    n_true = sub['Transported'].sum()
    print(f'  {fname:40s} | True: {n_true} | False: {len(sub)-n_true}')
    return fname

print('Saving submissions:\n')

# Stacking with optimized threshold (BEST)
save_submission('stack_opt', final_meta_test, BEST_THRESH)

# Stacking with default threshold
save_submission('stack_default', final_meta_test, 0.5)

# Simple average of all 8 models
avg_test = np.mean(test_list, axis=0)
save_submission('avg_8models', avg_test, 0.5)

# Simple average with optimized threshold
save_submission('avg_8models_opt', avg_test, BEST_THRESH)

# Best individual models
for name, test_p in zip(model_names, test_list):
    save_submission(name.lower(), test_p, 0.5)

print('\n\u2713 All submissions saved.')
print('\n\U0001f3c6 Submit submission_stack_opt.csv first!')

---
## 15. Final Summary

In [ ]:
# Final comparison table
print('=' * 55)
print('FINAL MODEL COMPARISON')
print('=' * 55)

all_results = []
for name, oof in zip(model_names, oof_list):
    acc = accuracy_score(y, (oof > 0.5).astype(int))
    all_results.append({'Model': name, 'CV Accuracy': acc})

all_results.append({'Model': 'Simple Avg (8)', 'CV Accuracy': avg_acc})
all_results.append({'Model': 'Stacking (CV)', 'CV Accuracy': stack_cv})
all_results.append({'Model': f'Stacking (t={BEST_THRESH:.3f})',
                    'CV Accuracy': accuracy_score(y, (meta_oof > BEST_THRESH).astype(int))})

results_df = pd.DataFrame(all_results).sort_values('CV Accuracy', ascending=False)
results_df['CV Accuracy'] = results_df['CV Accuracy'].map('{:.4f}'.format)
print(results_df.to_string(index=False))

print(f'\n\nv1 best LB: 0.80827 (tuned LightGBM)')
print(f'v2 target:  beat that with 8-model stacking + threshold optimization')
print(f'\nUpload submission_stack_opt.csv to Kaggle and compare!')